In [0]:
%run ../00_common/data_utils

In [0]:
%run ../00_common/UDF_utils

In [0]:
# ==============================
# 市场特性配置（列级别条件 — 所有市场在同一批次中混合处理）
# ==============================
# 启用Cid Mapping
MARKETS_ENABLE_CID_MATCH = ["JPN", "TWN"]

# 额外媒体类型 LINE 媒体类型
MARKETS_ENABLE_LINE_MEDIA     = ["JPN"]

# 启用local name 2 进行match
MARKETS_ENABLE_LOCALNAME2     = ["JPN", "KOR"]         

In [0]:
def get_RL_clear_consumer_df(task_id):

    # batch data 筛选特定market 和rakuten、linefift来源数据
    rakuten_linegift_sources = spark.table(f"{get_env_config('config_database')}.t_merge_exclude_consumer_config") \
                            .filter(F.col("tmec_type").isin([RAKUTEN, LINEGIFT])).select("tmec_sourcesystemcode").distinct().collect()

    rakuten_linegift_codes = [row.tmec_sourcesystemcode for row in rakuten_linegift_sources]

    t_clean_consumer = (
        spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer")
        .filter(F.col("task_id") == task_id)
        .filter(F.col("is_include") == True)
        .filter(F.col("srcc_mrkt_code").isin(MARKETS_ENABLE_CID_MATCH))
        .filter((F.col("SRCC_SRCS_CODE").isin(rakuten_linegift_codes)) | (F.col("SRCC_SRCS_CODE").startswith("50twn") & F.col("SRCC_SRCS_CODE").endswith("lngftund")))
    )

    return t_clean_consumer

In [0]:
def get_RL_master_consumer_df():
  # batch data 筛选特定market 和rakuten、linefift来源数据
  rakuten_linegift_sources = spark.table(f"{get_env_config('config_database')}.t_merge_exclude_consumer_config") \
                          .filter(F.col("tmec_type").isin([RAKUTEN, LINEGIFT])).select("tmec_sourcesystemcode").distinct().collect()

  rakuten_linegift_codes = [row.tmec_sourcesystemcode for row in rakuten_linegift_sources]

  t_master_consumer = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
        .filter(F.col("scon_mrkt_code").isin(MARKETS_ENABLE_CID_MATCH))
        .filter((F.col("SCON_SRCS_CODE").isin(rakuten_linegift_codes)) | (F.col("SCON_SRCS_CODE").startswith("50twn") & F.col("SCON_SRCS_CODE").endswith("lngftund")))
        .withColumn("record_type", F.lit(CID_MATCH_RECORD_TYPE_CM))
        .withColumn("master_recode_create_time", F.col("scon_creation_dt"))
    )
  
  return t_master_consumer